In [ ]:
%pip install --quiet --upgrade pip
%pip install --quiet openai httpx

In [4]:
import os

try:
    from utils import (
        generate_with_single_input,
        generate_with_multiple_input,
        get_proxy_url,
        get_proxy_headers,
        get_together_key,
    )
except ImportError:
    import httpx
    from openai import OpenAI, DefaultHttpxClient

    def get_together_key():
        return os.getenv("TOGETHER_API_KEY", "dummy-key")

    def get_proxy_url():
        return os.getenv("TOGETHER_PROXY_URL", "https://api.together.xyz/v1")

    def get_proxy_headers():
        return {}

    def _build_client():
        transport = httpx.HTTPTransport(verify=False)
        http_client = DefaultHttpxClient(transport=transport, headers=get_proxy_headers())
        return OpenAI(
            api_key=get_together_key(),
            base_url=get_proxy_url(),
            http_client=http_client,
        )

    def generate_with_single_input(prompt, role="user", max_tokens=256, model="Qwen/Qwen3.5-9B", **kwargs):
        client = _build_client()
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": role, "content": prompt}],
            max_tokens=max_tokens,
            extra_body={"reasoning": False},
            **kwargs,
        )
        message = response.choices[0].message
        return {"role": message.role, "content": message.content}

    def generate_with_multiple_input(messages, max_tokens=256, model="Qwen/Qwen3.5-9B", **kwargs):
        client = _build_client()
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=max_tokens,
            extra_body={"reasoning": False},
            **kwargs,
        )
        message = response.choices[0].message
        return {"role": message.role, "content": message.content}

print("Notebook utilities ready.")

Notebook utilities ready.
